# Patched Neural Field Diffusion on Synthetic Data

Test the new `PatchedNeuralFieldDiffusion` architecture on synthetic toy data.

**Key Innovation**: Space-filling curve serialization + per-patch neural fields
- Morton curve preserves 3D locality in 1D ordering
- Each patch (16 points) gets its own context → its own MLP
- Distance-weighted blending for smooth transitions

This should solve the global pooling collapse problem and enable learning
multi-modal geometry (multi-sphere, chairs with separate parts).

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm.notebook import tqdm
import time

# Our modules
from src.models.patched_neural_field import PatchedNeuralFieldDiffusion
from src.models.neural_field import NeuralFieldDiffusion
from src.diffusion.flow_matching import FlowMatchingLoss, FlowMatchingSampler
from data.toy_data import (
    generate_torus, generate_sphere, generate_helix,
    generate_multi_sphere_cube, generate_multi_sphere_ring,
    generate_two_spheres, generate_four_spheres,
    get_all_generators
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 1. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Data
SHAPES = ['multi_sphere_ring']  # Try: 'two_spheres', 'four_spheres', 'multi_sphere_cube', 'torus'
PATCH_SIZE = 16
N_POINTS = 512                  # Must be divisible by PATCH_SIZE
N_SAMPLES = 1000                # Training samples

# Model
HIDDEN_SIZE = 128
HIDDEN_SIZE_X = 32
NUM_HEADS = 4
NUM_BLOCKS = 4
NUM_NERF_BLOCKS = 2
NERF_MLP_RATIO = 2
MAX_FREQS = 6
BLEND_TEMPERATURE = 0.1

# Training
EPOCHS = 300
BATCH_SIZE = 32
LR = 1e-4

# Verify
assert N_POINTS % PATCH_SIZE == 0, f"N_POINTS must be divisible by PATCH_SIZE"
print(f"Shapes: {SHAPES}")
print(f"Points: {N_POINTS}, Patches: {N_POINTS // PATCH_SIZE} x {PATCH_SIZE}")

## 2. Dataset

In [ ]:
class ToyDataset(torch.utils.data.Dataset):
    """Simple toy dataset with pre-generated samples."""
    
    def __init__(self, shapes, n_points, n_samples):
        self.generators = get_all_generators()
        self.shape_names = shapes
        self.n_points = n_points
        
        # Pre-generate samples
        self.samples = []
        samples_per_shape = n_samples // len(shapes)
        
        for shape_name in shapes:
            gen_func = self.generators[shape_name]
            for _ in range(samples_per_shape):
                pc = gen_func(n_points)
                pc = pc.normalize()
                self.samples.append(torch.tensor(pc.points, dtype=torch.float32))
        
        print(f"Generated {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]


# Create dataset
dataset = ToyDataset(SHAPES, N_POINTS, N_SAMPLES)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True
)

In [ ]:
# Visualize training samples
fig = plt.figure(figsize=(16, 4))

for i in range(4):
    sample = dataset[i * 100].numpy()
    
    ax = fig.add_subplot(1, 4, i + 1, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2],
               c=colors, cmap='viridis', s=5, alpha=0.7)
    ax.set_title(f'Sample {i+1}')
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle(f'Training Samples: {SHAPES}', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Create Models (Patched vs Original)

In [ ]:
# Patched model (NEW)
patched_model = PatchedNeuralFieldDiffusion(
    in_channels=3,
    out_channels=3,
    hidden_size=HIDDEN_SIZE,
    hidden_size_x=HIDDEN_SIZE_X,
    num_heads=NUM_HEADS,
    num_cond_blocks=NUM_BLOCKS,      # DiT blocks (Stage 1) - renamed from num_blocks
    num_nerf_blocks=NUM_NERF_BLOCKS,  # NerfBlocks (Stage 2)
    nerf_mlp_ratio=NERF_MLP_RATIO,
    max_freqs=MAX_FREQS,
    patch_size=PATCH_SIZE,
    blend_temperature=BLEND_TEMPERATURE,
).to(DEVICE)

# Original model for comparison
original_model = NeuralFieldDiffusion(
    in_channels=3,
    out_channels=3,
    hidden_size=HIDDEN_SIZE,
    hidden_size_x=HIDDEN_SIZE_X,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    num_cond_blocks=NUM_BLOCKS // 2,
    nerf_mlp_ratio=NERF_MLP_RATIO,
    max_freqs=MAX_FREQS,
).to(DEVICE)

print(f"Patched model: {sum(p.numel() for p in patched_model.parameters()):,} params")
print(f"Original model: {sum(p.numel() for p in original_model.parameters()):,} params")

# Test forward pass
with torch.no_grad():
    test_x = torch.randn(2, N_POINTS, 3, device=DEVICE)
    test_t = torch.rand(2, device=DEVICE)
    
    out_patched = patched_model(test_x, test_t)
    out_original = original_model(test_x, test_t)
    
    print(f"\nPatched output: {out_patched.shape}")
    print(f"Original output: {out_original.shape}")

## 4. Training

In [ ]:
def train_model(model, dataloader, epochs, lr, device, name="Model"):
    """Train a model and return loss history."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = FlowMatchingLoss(schedule_type='linear')
    
    losses = []
    pbar = tqdm(range(epochs), desc=name)
    
    for epoch in pbar:
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        
        for batch in dataloader:
            x0 = batch.to(device)
            
            optimizer.zero_grad()
            output = loss_fn(model, x0)
            loss = output['loss']
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    return losses

In [ ]:
# Train PATCHED model
print("="*60)
print("Training PATCHED model...")
print("="*60)
patched_losses = train_model(patched_model, dataloader, EPOCHS, LR, DEVICE, "Patched")

In [ ]:
# Train ORIGINAL model for comparison
print("="*60)
print("Training ORIGINAL model...")
print("="*60)
original_losses = train_model(original_model, dataloader, EPOCHS, LR, DEVICE, "Original")

In [ ]:
# Compare training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(patched_losses, label='Patched', color='blue')
plt.plot(original_losses, label='Original', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(patched_losses[20:], label='Patched', color='blue')
plt.plot(original_losses[20:], label='Original', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (after warmup)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Loss - Patched: {patched_losses[-1]:.4f}, Original: {original_losses[-1]:.4f}")

## 5. Generate Samples

In [ ]:
def generate_samples(model, n_samples=4, n_points=512, n_steps=100, device='cpu'):
    """Generate samples using Euler integration."""
    model.eval()
    sampler = FlowMatchingSampler(model)
    noise = torch.randn(n_samples, n_points, 3, device=device)
    
    with torch.no_grad():
        samples = sampler.sample_euler(noise, n_steps=n_steps)
    
    return samples.cpu().numpy()

In [ ]:
# Generate samples from both models
print("Generating samples from PATCHED model...")
patched_samples = generate_samples(patched_model, n_samples=4, n_points=N_POINTS, n_steps=100, device=DEVICE)

print("Generating samples from ORIGINAL model...")
original_samples = generate_samples(original_model, n_samples=4, n_points=N_POINTS, n_steps=100, device=DEVICE)

print(f"Patched samples: {patched_samples.shape}")
print(f"Original samples: {original_samples.shape}")

In [ ]:
# Compare generated samples
fig = plt.figure(figsize=(16, 12))

# Row 1: Ground Truth
for i in range(4):
    gt = dataset[i * 100].numpy()
    ax = fig.add_subplot(3, 4, i + 1, projection='3d')
    colors = gt[:, 2]
    ax.scatter(gt[:, 0], gt[:, 1], gt[:, 2], c=colors, cmap='viridis', s=3, alpha=0.7)
    ax.set_title(f'GT {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

# Row 2: Patched Model
for i in range(4):
    sample = patched_samples[i]
    ax = fig.add_subplot(3, 4, i + 5, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=colors, cmap='plasma', s=3, alpha=0.7)
    ax.set_title(f'Patched {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

# Row 3: Original Model
for i in range(4):
    sample = original_samples[i]
    ax = fig.add_subplot(3, 4, i + 9, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=colors, cmap='coolwarm', s=3, alpha=0.7)
    ax.set_title(f'Original {i+1}')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])

plt.suptitle(f'{SHAPES}: GT (top) vs Patched (middle) vs Original (bottom)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Test Different Shapes

In [ ]:
# Quick test on simpler shapes to verify the architecture works
test_shapes = ['two_spheres', 'four_spheres', 'torus']

fig = plt.figure(figsize=(12, 4))

generators = get_all_generators()
for i, shape_name in enumerate(test_shapes):
    gen_func = generators[shape_name]
    pc = gen_func(N_POINTS)
    pc = pc.normalize()
    points = pc.points
    
    ax = fig.add_subplot(1, 3, i + 1, projection='3d')
    colors = points[:, 2]
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, cmap='viridis', s=3, alpha=0.7)
    ax.set_title(shape_name)
    ax.set_xlim([-1.2, 1.2])
    ax.set_ylim([-1.2, 1.2])
    ax.set_zlim([-1.2, 1.2])

plt.suptitle('Test Shapes for Debugging', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Space-Filling Curve Visualization

In [ ]:
from src.models.patched_neural_field import sort_by_space_filling_curve, patchify_points

# Visualize how Morton curve orders points
sample = dataset[0].unsqueeze(0)  # [1, N, 3]
sorted_sample, indices = sort_by_space_filling_curve(sample)
patches, centers, original_n = patchify_points(sorted_sample, PATCH_SIZE)

fig = plt.figure(figsize=(16, 4))

# Original order
ax1 = fig.add_subplot(1, 4, 1, projection='3d')
points = sample[0].numpy()
colors = np.arange(len(points))  # Color by original index
ax1.scatter(points[:, 0], points[:, 1], points[:, 2], c=colors, cmap='viridis', s=5)
ax1.set_title('Original Order')

# Morton-sorted order (use original_n to exclude padding)
ax2 = fig.add_subplot(1, 4, 2, projection='3d')
sorted_pts = sorted_sample[0, :original_n].numpy()
colors = np.arange(len(sorted_pts))
ax2.scatter(sorted_pts[:, 0], sorted_pts[:, 1], sorted_pts[:, 2], c=colors, cmap='viridis', s=5)
ax2.set_title('Morton Curve Order')

# Patches colored
ax3 = fig.add_subplot(1, 4, 3, projection='3d')
all_patch_pts = patches[0].reshape(-1, 3).numpy()[:original_n]
patch_colors = np.repeat(np.arange(patches.shape[1]), PATCH_SIZE)[:original_n]
ax3.scatter(all_patch_pts[:, 0], all_patch_pts[:, 1], all_patch_pts[:, 2], c=patch_colors, cmap='tab20', s=5)
ax3.set_title(f'Patches ({patches.shape[1]} patches)')

# Patch centers
ax4 = fig.add_subplot(1, 4, 4, projection='3d')
ax4.scatter(all_patch_pts[:, 0], all_patch_pts[:, 1], all_patch_pts[:, 2], c='lightgray', s=2, alpha=0.3)
center_pts = centers[0].numpy()
ax4.scatter(center_pts[:, 0], center_pts[:, 1], center_pts[:, 2], c='red', s=50, marker='o')
ax4.set_title('Patch Centers (red)')

plt.suptitle('Space-Filling Curve Visualization', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Points: {original_n}, Patches: {patches.shape[1]}, Patch size: {PATCH_SIZE}")

## 9. Summary

### Expected Results

**If Patched model works better:**
- Loss should decrease below 0.2 (vs ~0.4 for original)
- Generated samples should show distinct spheres/parts
- Multi-sphere should look like multiple spheres, not a blob

**If both models fail:**
- Try simpler shapes first: `two_spheres` → `four_spheres` → `multi_sphere_ring`
- May need more epochs or larger model

### Key Differences

| Aspect | Original | Patched |
|--------|----------|----------|
| Context | Global mean pooling | Per-patch context |
| MLP | Same for all points | Different per patch |
| Locality | Lost | Preserved via Morton curve |
| Multi-modal | Collapses modes | Separates modes |
| Super-resolution | Not supported cleanly | Native support via query_field |

## 8. Super-Resolution Generation (Patched Model Only)

The patched architecture enables **resolution-independent generation** through:
1. Extract patch context from a reference noise sample at training resolution
2. Query the blended neural field at **any number of points**
3. Distance-weighted blending ensures smooth output at any resolution

This is unique to the patched model - the original model cannot do this cleanly.

In [ ]:
def sample_superres_patched(model, n_points_base, n_points_target, n_steps=100, device='cpu'):
    """
    Generate samples at higher resolution than training using patched model.
    
    Strategy:
    1. Start with noise at base resolution (training size)
    2. At each ODE step, get patch context from current points
    3. Query the neural field at MORE points (upsampled noise)
    4. Integrate the higher-res velocity field
    
    Args:
        model: PatchedNeuralFieldDiffusion model
        n_points_base: Training resolution (e.g., 512)
        n_points_target: Target resolution (e.g., 2048, 4096)
        n_steps: Number of ODE integration steps
        device: torch device
    
    Returns:
        High-resolution point cloud [1, n_points_target, 3]
    """
    model.eval()
    
    # Start with noise at TARGET resolution
    x = torch.randn(1, n_points_target, 3, device=device)
    
    # Also track a base-resolution version for context
    x_base = torch.randn(1, n_points_base, 3, device=device)
    
    dt = 1.0 / n_steps
    
    with torch.no_grad():
        for step in range(n_steps):
            t = torch.tensor([step / n_steps], device=device)
            
            # Get patch context from base-resolution points
            # Returns tuple: (patch_context, patch_centers)
            patch_context, patch_centers = model.get_context(x_base, t)
            
            # Query field at high-resolution points
            v_target = model.query_field(x, t, patch_context, patch_centers)
            
            # Also get velocity for base points to keep them in sync
            v_base = model.query_field(x_base, t, patch_context, patch_centers)
            
            # Euler step
            x = x + v_target * dt
            x_base = x_base + v_base * dt
    
    return x.cpu()


def sample_superres_patched_v2(model, n_points_target, n_steps=100, device='cpu'):
    """
    Simpler super-resolution: just run ODE at target resolution.
    
    The patched model handles variable point counts natively through:
    - Morton sorting adapts to any N
    - Patchification pads to nearest multiple
    - Blending works for any query point
    
    This is cleaner than maintaining two point clouds.
    """
    model.eval()
    
    x = torch.randn(1, n_points_target, 3, device=device)
    dt = 1.0 / n_steps
    
    with torch.no_grad():
        for step in range(n_steps):
            t = torch.tensor([step / n_steps], device=device)
            v = model(x, t)
            x = x + v * dt
    
    return x.cpu()


print("Super-resolution sampling functions defined.")

In [ ]:
# Test super-resolution at multiple scales
RESOLUTIONS = [512, 1024, 2048, 4096]

print("Generating at multiple resolutions (patched model)...")
superres_samples = {}

for res in RESOLUTIONS:
    print(f"  Generating at {res} points...", end=" ")
    start = time.time()
    
    # Use v2 (simpler, direct generation at target resolution)
    sample = sample_superres_patched_v2(patched_model, res, n_steps=100, device=DEVICE)
    superres_samples[res] = sample[0].numpy()
    
    print(f"done in {time.time() - start:.2f}s")

print("\nAll resolutions generated!")

In [ ]:
# Visualize super-resolution results
fig = plt.figure(figsize=(16, 8))

for i, res in enumerate(RESOLUTIONS):
    sample = superres_samples[res]
    
    # Top row: 3D view
    ax = fig.add_subplot(2, len(RESOLUTIONS), i + 1, projection='3d')
    colors = sample[:, 2]
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], 
               c=colors, cmap='viridis', s=max(1, 10 - i*2), alpha=0.7)
    ax.set_title(f'{res} points')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.set_zlim([-1.5, 1.5])
    
    # Bottom row: Top-down view (XY plane)
    ax2 = fig.add_subplot(2, len(RESOLUTIONS), i + 1 + len(RESOLUTIONS))
    ax2.scatter(sample[:, 0], sample[:, 1], 
                c=sample[:, 2], cmap='viridis', s=max(0.5, 5 - i), alpha=0.5)
    ax2.set_xlim([-1.5, 1.5])
    ax2.set_ylim([-1.5, 1.5])
    ax2.set_aspect('equal')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_title(f'XY projection')

plt.suptitle(f'Super-Resolution: {SHAPES} (Trained at {N_POINTS} pts)', fontsize=14)
plt.tight_layout()
plt.show()

# Print statistics
print("\nResolution Statistics:")
print("-" * 50)
for res in RESOLUTIONS:
    sample = superres_samples[res]
    print(f"{res:5d} pts: mean={sample.mean(axis=0)}, std={sample.std():.4f}")

### 8.1 Context-Based Super-Resolution

Alternative approach: Use context from a **generated sample** at training resolution,
then query the neural field at arbitrary points.

This allows "upsampling" an existing generation.

In [ ]:
def upsample_point_cloud(model, base_sample, n_points_target, n_refine_steps=50, device='cpu'):
    """
    Upsample an existing point cloud using the patched model's neural field.
    
    Strategy:
    1. Get context from the base sample (defines the shape)
    2. Initialize target points by interpolating/jittering base points
    3. Refine using the neural field at t≈0 (near-manifold refinement)
    
    Args:
        model: PatchedNeuralFieldDiffusion model
        base_sample: [N_base, 3] numpy array of points
        n_points_target: Target number of points
        n_refine_steps: Steps to refine upsampled points
        device: torch device
    
    Returns:
        Upsampled point cloud [n_points_target, 3]
    """
    model.eval()
    
    # Convert base sample to tensor
    x_base = torch.tensor(base_sample, dtype=torch.float32, device=device).unsqueeze(0)
    n_base = x_base.shape[1]
    
    with torch.no_grad():
        # Get context at t=0 (final/clean state)
        # Returns tuple: (patch_context, patch_centers)
        t_final = torch.tensor([0.0], device=device)
        patch_context, patch_centers = model.get_context(x_base, t_final)
        
        # Initialize target points by:
        # 1. Include all base points
        # 2. Add jittered copies to reach target count
        n_extra = n_points_target - n_base
        
        if n_extra > 0:
            # Sample random base points and add small noise
            indices = torch.randint(0, n_base, (1, n_extra), device=device)
            extra_points = x_base[0, indices[0]] + 0.05 * torch.randn(n_extra, 3, device=device)
            x_target = torch.cat([x_base, extra_points.unsqueeze(0)], dim=1)
        else:
            x_target = x_base
        
        # Refine upsampled points using small t values (near manifold)
        # This "snaps" noisy points to the learned manifold
        for step in range(n_refine_steps):
            # Use small t to get near-manifold velocity
            t_refine = torch.tensor([0.05 * (1 - step / n_refine_steps)], device=device)
            v = model.query_field(x_target, t_refine, patch_context, patch_centers)
            
            # Small step size for refinement
            dt = 0.02
            x_target = x_target + v * dt
    
    return x_target[0].cpu().numpy()


# Test upsampling on a generated sample
print("Testing context-based upsampling...")

# First generate at training resolution
base_sample = superres_samples[512]  # Use 512-point sample as base
print(f"Base sample: {base_sample.shape[0]} points")

# Upsample to different resolutions
UPSAMPLE_TARGETS = [1024, 2048, 4096]
upsampled = {512: base_sample}

for target in UPSAMPLE_TARGETS:
    print(f"  Upsampling to {target} points...", end=" ")
    start = time.time()
    upsampled[target] = upsample_point_cloud(
        patched_model, base_sample, target, 
        n_refine_steps=30, device=DEVICE
    )
    print(f"done in {time.time() - start:.2f}s")

print("\nUpsampling complete!")

In [ ]:
# Compare direct generation vs upsampling
fig = plt.figure(figsize=(16, 12))

resolutions = [512, 1024, 2048, 4096]

for i, res in enumerate(resolutions):
    # Row 1: Direct generation at each resolution
    sample_direct = superres_samples[res]
    ax = fig.add_subplot(3, 4, i + 1, projection='3d')
    colors = sample_direct[:, 2]
    ax.scatter(sample_direct[:, 0], sample_direct[:, 1], sample_direct[:, 2],
               c=colors, cmap='viridis', s=max(1, 8 - i*2), alpha=0.7)
    ax.set_title(f'Direct: {res} pts')
    ax.set_xlim([-1.5, 1.5]); ax.set_ylim([-1.5, 1.5]); ax.set_zlim([-1.5, 1.5])
    
    # Row 2: Upsampled from 512
    sample_up = upsampled[res]
    ax = fig.add_subplot(3, 4, i + 5, projection='3d')
    colors = sample_up[:, 2]
    ax.scatter(sample_up[:, 0], sample_up[:, 1], sample_up[:, 2],
               c=colors, cmap='plasma', s=max(1, 8 - i*2), alpha=0.7)
    ax.set_title(f'Upsampled: {res} pts')
    ax.set_xlim([-1.5, 1.5]); ax.set_ylim([-1.5, 1.5]); ax.set_zlim([-1.5, 1.5])
    
    # Row 3: XY projection comparison
    ax = fig.add_subplot(3, 4, i + 9)
    ax.scatter(sample_direct[:, 0], sample_direct[:, 1], 
               c='blue', s=1, alpha=0.3, label='Direct')
    ax.scatter(sample_up[:, 0], sample_up[:, 1], 
               c='red', s=1, alpha=0.3, label='Upsampled')
    ax.set_xlim([-1.5, 1.5]); ax.set_ylim([-1.5, 1.5])
    ax.set_aspect('equal')
    ax.set_title(f'Overlay: {res} pts')
    if i == 0:
        ax.legend(loc='upper right', markerscale=5)

plt.suptitle(f'Direct Generation vs Upsampling from 512 pts\n{SHAPES}', fontsize=14)
plt.tight_layout()
plt.show()

### 8.2 Super-Resolution Summary

**Two approaches for generating at higher resolution:**

| Method | How it Works | Pros | Cons |
|--------|--------------|------|------|
| **Direct Generation** | Run ODE from noise at target resolution | True i.i.d. samples, cleaner | Different random seed each time |
| **Upsampling** | Use context from base sample, jitter + refine | Preserves base shape, faster | May have artifacts near jittered pts |

**Why this works for Patched model:**
- Morton sorting adapts to any point count N
- Patchification pads automatically (returns original_n for truncation)
- `query_field()` blends patch NFs based on distance → smooth interpolation
- Per-patch context means local geometry is preserved at any resolution

**Original model cannot do this cleanly** because:
- Global mean pooling creates fixed-size context
- NerfBlocks apply same MLP to all points
- No locality-aware blending mechanism